# E3 — cgroup CPU Cap Analysis (Headline L4 Scenario)

**Experiment:** E3-cpu-cap  
**Layer:** L4 — compose resource limit (`deploy.resources.limits.cpus`)  
**Scenarios:** S0-baseline vs S7-cpu-cap-half (`BACK_CPUS=0.5`)  
**Workload:** W4-mixed (diurnal shape, ~26 min per run)

## Research question

When the backend container is throttled to 0.5 CPUs via cgroup:
1. Does `container_cpu_cfs_throttled_seconds_total > 0.5/s` (sustained) fire as the unique fingerprint?
2. Does the recommender correctly attribute this to **L4** (not L1 heap or L3 config)?
3. The host CPU is **not** saturated — distinguishing this from a 'bigger node' problem.

This is the **headline L4 scenario** — closest analogue to CherryPick-style instance-type selection.
The burst phase of W4 (users × 2 for 1 min) exposes the throttling most clearly.

## Figures produced
- **Fig 1** — CPU throttling rate time series against the W4 diurnal shape (S0 vs S7)
- **Fig 2** — p99 latency across the diurnal phases
- **Fig 3** — Throughput under the diurnal shape (requests/s vs time)

Run `python experiments/E3-cpu-cap/run.py --intensities medium` first.

In [ ]:
import json
import csv
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
import pandas as pd
from IPython.display import display

matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

REPO_ROOT = Path('.').resolve().parent.parent
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = Path('.') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

EXPERIMENT_KEY = 'E3-cpu-cap'
SCENARIOS   = ['S0-baseline', 'S7-cpu-cap-half']
INTENSITIES = ['low', 'medium', 'high']

COLORS = {
    'S0-baseline':    '#2196F3',
    'S7-cpu-cap-half':'#F44336',
}
LABELS = {
    'S0-baseline':    'S0 baseline (cpus=2.0)',
    'S7-cpu-cap-half':'S7 cpu-cap-half (cpus=0.5)',
}

# W4 diurnal phase boundaries (seconds from start)
W4_PHASES = [
    (0,    300,  'ramp-up'),
    (300,  900,  'plateau'),
    (900,  960,  'burst'),
    (960,  1200, 'settle'),
    (1200, 1500, 'ramp-down'),
]

In [ ]:
def load_locust_stats(run_dir: Path) -> dict:
    f = run_dir / 'locust_stats.csv'
    if not f.exists():
        return {}
    rows = {}
    with open(f) as fh:
        for row in csv.DictReader(fh):
            name = row.get('Name', '')
            try:
                total = float(row.get('Request Count', 1) or 1)
                rows[name] = {
                    'p95':        float(row.get('95%', 0) or 0),
                    'p99':        float(row.get('99%', 0) or 0),
                    'error_rate': float(row.get('Failure Count', 0) or 0) / max(1, total),
                    'rps':        float(row.get('Requests/s', 0) or 0),
                }
            except (ValueError, TypeError):
                pass
    return rows


def load_locust_history(run_dir: Path) -> pd.DataFrame:
    """Load per-second history from locust_stats_history.csv for time-series plots."""
    f = run_dir / 'locust_stats_history.csv'
    if not f.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(f)
        df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')
        return df.dropna(subset=['Timestamp'])
    except Exception:
        return pd.DataFrame()


def get_prom_series(prom: dict, key: str):
    data = prom.get(key, {})
    if data.get('status') != 'success':
        return [], []
    ts_list, v_list = [], []
    for series in data.get('data', {}).get('result', []):
        for ts, v in series.get('values', []):
            try:
                ts_list.append(float(ts))
                v_list.append(float(v))
            except (ValueError, TypeError):
                pass
    return ts_list, v_list


def load_run(run_id: str) -> dict:
    d = RESULTS_DIR / run_id
    meta = yaml.safe_load((d / 'metadata.yaml').read_text()) if (d / 'metadata.yaml').exists() else {}
    prom = json.loads((d / 'prom_snapshot.json').read_text()) if (d / 'prom_snapshot.json').exists() else {}
    return {
        'run_id':  run_id,
        'meta':    meta,
        'prom':    prom,
        'stats':   load_locust_stats(d),
        'history': load_locust_history(d),
    }


def add_phase_bands(ax, t0: float):
    """Shade W4 diurnal phase bands behind a time-series axis."""
    phase_colors = {
        'ramp-up': '#E3F2FD', 'plateau': '#E8F5E9', 'burst': '#FFEBEE',
        'settle': '#FFF3E0', 'ramp-down': '#F3E5F5'
    }
    for start, end, label in W4_PHASES:
        ax.axvspan((t0 + start) / 60, (t0 + end) / 60,
                   alpha=0.25, color=phase_colors[label], label=f'phase: {label}')
    # deduplicate legend
    handles, labels_ = ax.get_legend_handles_labels()
    seen = {}
    unique = [(h, l) for h, l in zip(handles, labels_) if not (l in seen or seen.update({l: True}))]
    ax.legend(*zip(*unique), fontsize=8, loc='upper right')

In [ ]:
runs = []
if RESULTS_DIR.exists():
    for run_dir in sorted(RESULTS_DIR.iterdir()):
        mf = run_dir / 'metadata.yaml'
        if not mf.exists():
            continue
        meta = yaml.safe_load(mf.read_text())
        if meta.get('experiment') == EXPERIMENT_KEY:
            runs.append(load_run(run_dir.name))

print(f'Found {len(runs)} run(s) for {EXPERIMENT_KEY}')
for r in runs:
    m = r['meta']
    print(f"  {r['run_id'][:8]}  scenario={m.get('scenario','?'):20s}  "
          f"intensity={m.get('intensity','?')}  workload={m.get('workload','?')}")

if not runs:
    print('\nNo results yet. Generate data with:')
    print('  python experiments/E3-cpu-cap/run.py --intensities medium')
    print('  (W4 diurnal runs ~26 min per cell)')

In [ ]:
records = []
for r in runs:
    m, prom, stats = r['meta'], r['prom'], r['stats']
    agg = stats.get('Aggregated', next(iter(stats.values()), {}))

    key_thr = "container_cpu_cfs_throttled_seconds_total{name='correctexam-back'}"
    key_per = "container_cpu_cfs_periods_total{name='correctexam-back'}"
    _, throttled = get_prom_series(prom, key_thr)
    _, periods   = get_prom_series(prom, key_per)

    # throttle_ratio = throttled_periods / total_periods (fraction of periods throttled)
    throttle_ratios = [t / p for t, p in zip(throttled, periods) if p > 0]

    records.append({
        'scenario':           m.get('scenario', '?'),
        'intensity':          m.get('intensity', '?'),
        'run_id':             r['run_id'][:8],
        'p99_ms':             agg.get('p99', 0),
        'p95_ms':             agg.get('p95', 0),
        'error_rate_%':       round(agg.get('error_rate', 0) * 100, 2),
        'rps':                agg.get('rps', 0),
        'throttle_mean':      round(float(np.mean(throttled)), 3)        if throttled       else 0,
        'throttle_ratio_mean':round(float(np.mean(throttle_ratios)), 3)  if throttle_ratios else 0,
    })

df = pd.DataFrame(records)
if not df.empty:
    display(df.sort_values(['scenario', 'intensity']))
else:
    print('No data yet.')

In [ ]:
# Figure 1 — CPU throttling rate over W4 diurnal shape (S0 vs S7, medium intensity)
# The burst phase (t=15-16 min) should trigger the strongest throttling in S7.

target = {s: None for s in SCENARIOS}
for r in runs:
    sc = r['meta'].get('scenario')
    it = r['meta'].get('intensity')
    if it == 'medium' and sc in target:
        target[sc] = r

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
fig.suptitle(
    'Fig 1 — E3 CPU Throttling: cAdvisor Signal vs Diurnal Shape (W4-mixed, medium intensity)',
    fontsize=12, fontweight='bold'
)

t0_ref = None
key_thr = "container_cpu_cfs_throttled_seconds_total{name='correctexam-back'}"

for scenario, r in target.items():
    c   = COLORS[scenario]
    lbl = LABELS[scenario]

    if r is None:
        for ax in axes:
            ax.text(0.5, 0.5, f'Run for {scenario}\nnot found',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=10, color='gray', style='italic')
        continue

    prom = r['prom']
    ts, throttled = get_prom_series(prom, key_thr)
    if ts and throttled:
        if t0_ref is None:
            t0_ref = ts[0]
        t_rel = [(t - t0_ref) / 60 for t in ts]
        axes[0].plot(t_rel, throttled, color=c, label=lbl, linewidth=2)

    hist = r['history']
    if not hist.empty and t0_ref is not None:
        t_hist = [(t - t0_ref) / 60 for t in hist['Timestamp']]
        if 'User count' in hist.columns:
            axes[1].plot(t_hist, hist['User count'], color=c, label=lbl, linewidth=2)

axes[0].axhline(0.5, color='orange', linestyle='--', linewidth=1.2, alpha=0.8,
                label='diagnostic threshold (0.5/s)')
axes[0].set_title('CPU CFS Throttled Seconds (cAdvisor)')
axes[0].set_ylabel('container_cpu_cfs_throttled_seconds_total')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Concurrent Users (W4 Diurnal Shape)')
axes[1].set_xlabel('Time into run (min)')
axes[1].set_ylabel('user count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Mark the burst phase
for ax in axes:
    ax.axvspan(900/60, 960/60, alpha=0.15, color='red', label='burst phase')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E3-fig1-cpu-throttling.pdf')
plt.show()
print('Saved: figures/E3-fig1-cpu-throttling.pdf')

In [ ]:
# Figure 2 — p99 latency and throughput by scenario × intensity (bar chart)

if df.empty:
    print('No data for Figure 2.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Fig 2 — E3 SLO Impact: p99 Latency and Throughput (S0 vs S7)',
                 fontsize=12, fontweight='bold')

    x     = np.arange(len(INTENSITIES))
    width = 0.35

    for idx, scenario in enumerate(SCENARIOS):
        sub = df[df['scenario'] == scenario].set_index('intensity')
        p99 = [sub.loc[i, 'p99_ms'] if i in sub.index else 0 for i in INTENSITIES]
        rps = [sub.loc[i, 'rps']    if i in sub.index else 0 for i in INTENSITIES]
        off = (idx - 0.5) * width
        axes[0].bar(x + off, p99, width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)
        axes[1].bar(x + off, rps, width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)

    for ax, title, ylabel in [
        (axes[0], 'p99 Latency (ms) — whole run', 'p99 latency (ms)'),
        (axes[1], 'Throughput (req/s) — whole run', 'requests/s'),
    ]:
        ax.set_title(title)
        ax.set_xlabel('Intensity (ceiling users)')
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(INTENSITIES)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'E3-fig2-slo-impact.pdf')
    plt.show()
    print('Saved: figures/E3-fig2-slo-impact.pdf')

In [ ]:
# Figure 3 — Throttle ratio vs p99 scatter + CherryPick comparison note
# Shows that the cAdvisor signal predicts p99 degradation and
# uniquely identifies L4 (host CPU is not saturated).

if df.empty:
    print('No data for Figure 3.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.suptitle('Fig 3 — E3 Evidence Link: CPU Throttle Ratio vs p99 Latency',
                 fontsize=12, fontweight='bold')

    for scenario in SCENARIOS:
        sub = df[df['scenario'] == scenario]
        ax.scatter(sub['throttle_ratio_mean'], sub['p99_ms'],
                   color=COLORS[scenario], label=LABELS[scenario], s=80, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(row['intensity'],
                        (row['throttle_ratio_mean'], row['p99_ms']),
                        textcoords='offset points', xytext=(4, 4), fontsize=9)

    ax.set_xlabel('Mean CPU Throttle Ratio (throttled_periods / total_periods)')
    ax.set_ylabel('p99 Latency (ms)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    note = (
        'CherryPick comparison: CherryPick selects instance type based on\n'
        'throughput/cost ratio alone, with no visibility into WHY performance\n'
        'degrades. The throttle_ratio signal here identifies the root cause\n'
        '(L4 cgroup limit) and proposes a targeted fix (raise BACK_CPUS),\n'
        'not a larger instance type.'
    )
    ax.text(0.02, 0.97, note, transform=ax.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'E3-fig3-evidence-link.pdf')
    plt.show()
    print('Saved: figures/E3-fig3-evidence-link.pdf')

## Interpretation

### Expected outcome (pre-run prediction)

| Metric | S0 baseline | S7 cpu-cap-half |
|--------|-------------|------------------|
| throttle_ratio_mean | ~0 | > 0.5 (sustained) |
| p99 latency | baseline | 3–5× baseline |
| throughput | baseline | plateau far below |

### Why this matters vs CherryPick

CherryPick-style baselines observe throughput/cost and recommend a bigger instance.  
Our recommender observes `container_cpu_cfs_throttled_seconds_total` and recommends  
a targeted IaC change (`BACK_CPUS=2.0` in the compose file), which:
- costs nothing if you already have the CPU headroom on the host
- is actionable in minutes (re-deploy), not hours (re-provision)
- is traceable to evidence (the throttle signal in the snapshot bundle)

### Diagnostic rule (from diagnose.py)
```
IF container_cpu_cfs_throttled_seconds_total > 0.5/s  (sustained 50% of samples)
THEN scenario = S7-cpu-cap-half, layer = L4, confidence = f(throttle_rate)
```

**Anti-confusion with L1 heap (S1):** Both S1 and S7 produce p99 spikes.  
S7's unique fingerprint is the cAdvisor throttle signal — absent in S1.  
S1's fingerprint is the heap ratio — absent in S7.